In [ ]:
import importlib

import browser_session
import helpers
import entities
import model_config

importlib.reload(browser_session)
importlib.reload(helpers)
importlib.reload(entities)
# importlib.reload(model_config)

from browser_session import BrowserSession
from helpers import Helpers
from entities import Element
from model_config import model, strong_model

from playwright.async_api import async_playwright
from langchain_core.tools import tool
import json
from dataclasses import dataclass
from typing import Any

from deepagents import create_deep_agent, DeepAgentState
from pywin.framework.toolmenu import tools
from sqlalchemy.sql.base import elements

browser_session = BrowserSession(headless=False)

SYSTEM_PROMPT = """
You are a web browser agent.
Follow the Observe -> Decide -> Act loop until the user's task is verified complete:
- Start with `navigate_and_observe` using the URL provided by the user.
- Decide the next safe actions from that observation.
- Act by delegating browser actions to `steps-executor` or viewport search to `page-navigator`.
- After the subagent returns, call `observe_page` and repeat the loop.

Given the user's task and the observed page state, determine the next interaction or sequence of interactions that makes useful progress toward the task.
Guidelines:
- Base all interactions only on the observed page state and provided context.
- Reference only elements and information present in the observation.
- Use element indices exactly as provided.
- Do not invent page content, elements, or available actions.
- Prefer the smallest useful sequence of interactions.
- You may return multiple interactions when they can all be determined from the current observed state.
- Do not plan interactions that depend on the result of an earlier interaction unless that result is already known.

After observing the page, use the `task` tool to delegate browser actions. Set `subagent_type` to `steps-executor`. Set the task tool's `description` argument to a JSON-serialized string containing exactly this structure:
{
  "steps": [
    {"action": "fill", "element_index": 6, "value": "text"},
    {"action": "fill", "element_index": 8, "value": "text"},
    {"action": "click", "element_index": 11}
  ]
}

If clicking an element opens a form, modal, menu, tab, or page, submit only
that click. Do not predict the elements that will appear afterward. Wait for
the next observation.

The `description` argument must contain only that JSON object. Do not summarize the steps in prose. Do not return the execution plan directly to the user instead of calling the `task` tool.
Use only the actions `click`, `fill`, `select`, or `press`. The `value` field is required for every action except `click`. Use exact indices from the latest observation, and make any page-changing action the final step. Do not include explanations, Markdown, an expected URL, or an observation ID in the JSON plan.

If the target element is not visible and more page content remains, call the `task` tool with `subagent_type` set to `page-navigator`. Set `description` to only a JSON object containing up to 5 target keywords:
{
  "keywords": ["target label", "related heading"]
}
After `page-navigator` returns, call `observe_page` to refresh the visible elements and their indices before deciding the next action.

Never assume an interaction succeeded or that the page changed unless explicitly indicated by a subsequent observation or provided context.
Use `navigate_and_observe` only to open the initial URL or when intentionally navigating to a known URL. Never use it to verify an action.
Always call `observe_page` after the executor returns. Do not treat the executor's confirmation as proof that the user's task succeeded. Only report completion when a subsequent page observation contains clear evidence that the requested outcome was achieved. If it was not achieved, decide and execute the next safe actions.
If the observation does not provide enough information, or no available tool can perform the next interaction, stop and clearly explain what is blocking progress instead of observing again or guessing.
"""


@tool
async def navigate_and_observe(url: str) -> str:
    """Navigate to a URL and return a snapshot of the resulting page."""
    return await browser_session.attempt_navigate_and_observe(url)


@tool
async def observe_page() -> str:
    """Return a snapshot of the current page without navigating."""
    return await browser_session.attempt_observe_page()


In [ ]:
from langchain.tools import tool


@tool
async def click(element_index: int) -> str:
    """Click an element in the browser by its element index."""
    await browser_session.click(element_index)
    return f"Clicked element [{element_index}]"


@tool
async def fill(element_index: int, value: str) -> str:
    """Fill an input element in the browser with a value."""
    await browser_session.fill(element_index, value)
    return f"Filled element [{element_index}]"


@tool
async def select(element_index: int, value: str) -> str:
    """Select an option from a browser select element."""
    await browser_session.select(element_index, value)
    return f"Selected an option in element [{element_index}]"


@tool
async def press(element_index: int, value: str) -> str:
    """Press a keyboard key or key combination on a browser element."""
    await browser_session.press(element_index, value)
    return f"Pressed {value} on element [{element_index}]"

In [ ]:
from langchain_quickjs import CodeInterpreterMiddleware

STEPS_EXECUTOR_SUBAGENT_SYSTEM_PROMPT = """
You are a web browser agent that executes the steps provided to you.
You have access to the following browser tools: click, fill, select, and press.
The task will contain a list of steps. Execute each step in order using the
corresponding tool and the provided arguments.
The task will look like this:
{
  "steps": [
    {"action": "click", "element_index": 4},
    {"action": "fill", "element_index": 1, "value": "text to enter"},
    {"action": "select", "element_index": 2, "value": "option_value"},
    {"action": "press", "element_index": 3, "value": "Enter"}
  ]
}

The eval tool supports Programmatic Tool Calling (PTC): JavaScript running
inside eval() can call the browser tools through tools.click(),
tools.fill(), tools.select(), and tools.press().
Prefer a single eval() call that executes all provided steps sequentially in
JavaScript. Keep intermediate tool results inside JavaScript variables rather
than returning them to the model between steps.
For each step:
- Call the tool whose name matches the "action".
- Pass "element_index" to the tool.
- If "value" is present, pass it as the value argument.
- Await each tool call before executing the next step.
- Execute the steps in the exact order provided.

Do not skip, reorder, modify, or invent steps.
After execution, report:
- The actions that completed, formatted as `action[element_index]`.
- The first failed action and its error, if any.
- Whether the final completed action may have changed the page.

Do not include values entered into fields.
Do not claim that the user's overall task succeeded; report only what actions ran.
"""

steps_executor = {
    "name": "steps-executor",
    "description": (
        "Execute browser interaction steps in order. The task description must be "
        "a JSON object containing a steps array; never accept or infer steps from prose."
    ),
    "system_prompt": STEPS_EXECUTOR_SUBAGENT_SYSTEM_PROMPT,
    "tools": [click, fill, select, press],
    "model": model,
    "middleware": [CodeInterpreterMiddleware(ptc=["click", "fill", "select", "press"])],
}


In [ ]:
@tool
async def get_text_in_viewport() -> str:
    """Return the text currently visible in the browser viewport."""
    return await browser_session.get_text_in_viewport()


@tool
async def scroll(amount: float) -> str:
    """Scroll by a multiple of the viewport height."""
    return await browser_session.scroll(amount)


PAGE_NAVIGATOR_SYSTEM_PROMPT = """
You are a web-page navigation agent. Your job is to locate a target element by inspecting the text currently visible in the viewport and scrolling the page up or down as needed.

The task contains a JSON object with up to 5 keywords associated with the target element:
{
  "keywords": ["target label", "related heading"]
}

The eval tool supports Programmatic Tool Calling (PTC). Use these exact camelCase signatures with one object argument:
- `await tools.getTextInViewport({})` to retrieve the text currently visible in the viewport.
- `await tools.scroll({ amount: 0.5 })` to move the page.

Prefer one `eval()` call for the navigation loop. Inside JavaScript, normalize the viewport text and keywords to lowercase and use direct keyword inclusion. Keep intermediate tool results inside JavaScript variables.

Navigation procedure:
1. Call `await tools.getTextInViewport({})`.
2. Normalize the returned text and keywords to lowercase.
3. Stop when the viewport contains any keyword.
4. If no keyword matches, call `await tools.scroll({ amount: 0.5 })`, then inspect the viewport again.
5. Repeat for at most 12 scroll operations. Stop early if scrolling no longer changes position.
6. If no exact match is found, return the final viewport text for the main agent to assess.

Valid scroll amounts are:
- `amount: 1` → scroll down one full viewport
- `amount: 0.5` → scroll down half a viewport
- `amount: -1` → scroll up one full viewport

Scrolling strategy:
- Prefer scrolling down when searching forward through the page.
- Use `amount: 0.5` for the default forward search.
- Scroll up only when the task clearly indicates that the target is above the current viewport.
- Avoid unnecessary scrolling, oscillating between the same regions, or revisiting viewport text that has already been checked.

Do not ask the user to approve a retry or choose a scrolling strategy. If the first eval call fails, retry once with the simple 0.5-viewport loop above. If the retry also fails, return the exact error to the main agent.
After searching, report whether a keyword was found and how many scroll operations were performed.
"""



page_navigator = {
    "name": "page-navigator",
    "description": (
        "Locate an off-screen target from a JSON keywords list by inspecting "
        "viewport text and scrolling the current page."
    ),
    "system_prompt": PAGE_NAVIGATOR_SYSTEM_PROMPT,
    "tools": [get_text_in_viewport, scroll],
    "model": model,
    "middleware": [CodeInterpreterMiddleware(ptc=["get_text_in_viewport", "scroll"])],
}

In [ ]:
agent = create_deep_agent(model=model, system_prompt=SYSTEM_PROMPT, tools=[navigate_and_observe, observe_page],
                          subagents=[steps_executor, page_navigator])

In [ ]:
# result = await agent.ainvoke({"messages": [{"role": "user",
#                                             "content": "Navigate to https://nexmenus.com/ and find the privacy policy link, open it, and then get the title"}]})
# print(result["messages"][-1].content)

In [ ]:
result = await agent.ainvoke({"messages": [{"role": "user",
                                            "content": "For each input data needed, generate random test data. Navigate to https://nexmenus.com/ and create a new account, then create a new Menu, then add a new Menu item and preview the Menu"}]})
print(result["messages"][-1].content)


In [ ]:
# await browser_session.close()